# 04 - IEEE-CIS LTN-Style Fraud Rule Analysis

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ["THESIS_QUICK_RUN"] = "0"
    os.environ["THESIS_SYNTHETIC_FALLBACK"] = "0"
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

Cloning into '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection'...


{'project_root': '/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection', 'git_commit': 'f50aaaa35b16c848247191114ccfb875641e2039', 'quick_run': False, 'synthetic_fallback': False, 'kaggle': True}


## Thiết lập

Rules và quantile thresholds chỉ fit trên train. Notebook đánh giá rule truth values,
class-balanced knowledge-base satisfaction và một diagnostic fuzzy predicate khả vi.
Đây là LTN-style explanation layer, không phải end-to-end LTN predictor.

In [2]:
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data
from src.explanation import rule_quality_table
from src.logic import FraudKnowledgeBase, FraudRuleEngine

config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
output_dir = OUTPUT_BASE / "04_ieee_cis_ltn_rule_analysis"
output_dir.mkdir(parents=True, exist_ok=True)
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
knowledge_base = FraudKnowledgeBase(engine)
print({"data_source": data_source, "active_rules": len(engine.rules), "skipped": engine.skipped_rules})
display(engine.fitted_thresholds())

{'data_source': 'real', 'active_rules': 5, 'skipped': {}}


,rule,feature,operator,configured_value,fitted_threshold
0,high_transaction_amount,TransactionAmt,greater_quantile,0.95,444.95
1,unusual_transaction_hour,transaction_hour,outside_range,"[6.0, 23.0]","[6.0, 23.0]"
2,high_amount_and_device_risk,TransactionAmt,greater_quantile,0.9,280.0
3,high_amount_and_device_risk,DeviceType,category_risk,0.7,0.7
4,high_amount_and_email_risk,TransactionAmt,greater_quantile,0.9,280.0
5,high_amount_and_email_risk,P_emaildomain,category_risk,0.7,0.7
6,high_velocity_proxy,C1,greater_quantile,0.95,22.0


## Differentiable fuzzy predicate diagnostic

In [3]:
import torch
from src.logic import SoftThresholdPredicate, TensorLogic

amount = torch.tensor(prepared.train_frame["TransactionAmt"].to_numpy(float), dtype=torch.float32)
center = torch.nanmedian(amount)
scale = torch.nan_to_num(amount.std(), nan=1.0).clamp_min(1e-6)
values = torch.nan_to_num((amount - center) / scale)
labels = torch.tensor(prepared.y_train, dtype=torch.float32)
predicate = SoftThresholdPredicate(float(torch.quantile(values, 0.90)), temperature=0.5, learnable=True)
optimizer = torch.optim.Adam(predicate.parameters(), lr=0.03)
initial_threshold = float(predicate.threshold.detach())
history = []
for _ in range(30 if QUICK_RUN else 100):
    optimizer.zero_grad()
    evidence = predicate(values)
    satisfaction = 0.5 * (
        TensorLogic.forall(evidence[labels == 1]) +
        TensorLogic.forall(1.0 - evidence[labels == 0])
    )
    loss = 1.0 - satisfaction + 0.01 * (predicate.threshold - initial_threshold).pow(2)
    loss.backward()
    optimizer.step()
    history.append(float(satisfaction.detach()))
tensor_demo = pd.DataFrame([{
    "initial_threshold": initial_threshold,
    "learned_threshold": float(predicate.threshold.detach()),
    "initial_satisfaction": history[0],
    "final_satisfaction": history[-1],
}])
display(tensor_demo.round(5))

,initial_threshold,learned_threshold,initial_satisfaction,final_satisfaction
0,0.88674,-0.12174,0.45732,0.47877


## Rule and knowledge-base results

In [4]:
split_frames = {
    "train": (prepared.train_frame, prepared.y_train),
    "validation": (prepared.validation_frame, prepared.y_validation),
    "test": (prepared.test_frame, prepared.y_test),
}
activation = float(config["logic"]["activation_threshold"])
quality_frames = []
satisfaction_rows = []
for split, (split_frame, labels) in split_frames.items():
    truth = engine.evaluate(split_frame)
    quality_frames.append(rule_quality_table(truth, labels, activation).assign(split=split))
    satisfaction_rows.append({"split": split, **knowledge_base.satisfaction_breakdown(split_frame, target)})
quality = pd.concat(quality_frames, ignore_index=True)
satisfaction = pd.DataFrame(satisfaction_rows)
display(quality.round(4), satisfaction.round(4))
quality.to_csv(output_dir / "ieee_rule_quality.csv", index=False)
satisfaction.to_csv(output_dir / "ieee_knowledge_base_satisfaction.csv", index=False)
engine.fitted_thresholds().to_csv(output_dir / "ieee_fitted_rule_thresholds.csv", index=False)
tensor_demo.to_csv(output_dir / "ieee_tensor_predicate_diagnostic.csv", index=False)

,rule,coverage,active_count,fraud_precision,lift,mean_truth,rule_auc,split
0,high_amount_and_device_risk,0.0014,593,0.3120,8.8707,0.0174,0.6494,train
1,high_velocity_proxy,0.0490,20249,0.0778,2.2131,0.0624,0.6169,train
2,high_transaction_amount,0.0424,17533,0.0422,1.2001,0.0727,0.4994,train
3,unusual_transaction_hour,0.1683,69551,0.0336,0.9546,0.5025,0.5196,train
4,high_amount_and_email_risk,0.0000,1,0.0000,0.0000,0.0028,0.6054,train
5,high_amount_and_device_risk,0.0008,75,0.6267,18.2481,0.0105,0.6678,validation
6,high_amount_and_email_risk,0.0000,2,0.5000,14.5597,0.0028,0.5770,validation
7,high_velocity_proxy,0.0542,4797,0.0930,2.7074,0.0680,0.5895,validation
8,high_transaction_amount,0.0418,3704,0.0613,1.7846,0.0719,0.5130,validation
9,unusual_transaction_hour,0.1622,14369,0.0314,0.9140,0.4979,0.5246,validation


,split,overall_satisfaction,positive_satisfaction,negative_satisfaction,balanced_satisfaction
0,train,0.4623,0.5790,0.4580,0.5185
1,validation,0.4654,0.6008,0.4606,0.5307
2,test,0.4696,0.5753,0.4658,0.5205


In [5]:
validation_quality = quality.query("split == 'validation'")
test_quality = quality.query("split == 'test'")
stability = validation_quality.merge(test_quality, on="rule", suffixes=("_validation", "_test"))
stability["coverage_delta"] = stability["coverage_test"] - stability["coverage_validation"]
stability["lift_delta"] = stability["lift_test"] - stability["lift_validation"]
display(stability[["rule", "coverage_delta", "lift_delta"]].round(4))
stability.to_csv(output_dir / "ieee_rule_stability.csv", index=False)

,rule,coverage_delta,lift_delta
0,high_amount_and_device_risk,-0.0000,0.6442
1,high_amount_and_email_risk,-0.0000,-14.5597
2,high_velocity_proxy,-0.0077,-0.5654
3,high_transaction_amount,0.0017,-0.1606
4,unusual_transaction_hour,-0.0109,0.0056


## Takeaways

In [6]:
strongest = test_quality.sort_values("lift", ascending=False).iloc[0]
test_satisfaction = satisfaction.query("split == 'test'").iloc[0]
display(Markdown(
    f"- Highest test rule lift: **{strongest['rule']} = {strongest['lift']:.3f}**.\n"
    f"- Test balanced knowledge-base satisfaction: **{test_satisfaction['balanced_satisfaction']:.3f}**.\n"
    "- Rule lift and satisfaction measure association/logic agreement, not causality."
))

- Highest test rule lift: **high_amount_and_device_risk = 18.892**.
- Test balanced knowledge-base satisfaction: **0.521**.
- Rule lift and satisfaction measure association/logic agreement, not causality.